### Oasis Clinical Utility Experiment with Raw and GS Transformed Dataset

The purpose of this notebook is to measure the impact of the GS transformations on the model's predictive performance on a realistic clinical settings. In this notebook we have utilized the OASIS-1 Dementia dataset to perform this experiment. We used binary classification settings for the model training. The models are trained using 5-fold CV under raw, GS 0, 10, 20, 30, 40, and 50% masking settings individually, and result are compared.

## Imports and Functions

In [1]:
# =============================================================================
# OASIS CLINICAL UTILITY - DEMENTIA CLASSIFICATION
# =============================================================================
# Binary classification: Healthy (0) vs Demented (1)
# 5-Fold Cross-Validation at subject level
# 
# Conditions tested:
#   - raw: Original images
#   - gs0, gs10, gs20, gs30, gs40, gs50: GradientShaping at various mask levels
#
# Evaluation modes:
#   - Adaptive: Train and eval on same condition
#   - Blind: Train on GS, eval on raw
# =============================================================================

import os
import sys
import time
import json
import pickle
import gc
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)

# Suppress deprecation warnings
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*deprecated.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.GradScaler.*deprecated.*")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Paths
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'data' / 'oasis' / 'processed' / 'utility'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'oasis' / 'utility'
MODELS_DIR = PROJECT_ROOT / 'models' / 'oasis' / 'utility'
CACHE_DIR = PROJECT_ROOT / 'cache' / 'oasis' / 'utility'

# -----------------------------------------------------------------------------
# Caching policy
# -----------------------------------------------------------------------------
# If False (default): GS tensors are built on-the-fly, nothing is cached to disk.
# If True: fold/split/condition tensors are cached for faster re-runs. THIS WILL CONSUME SIGNIFICANT DISK MEMORY!!!!
CACHE_ENABLED = False

# Create directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Add project root to path for gs_functions
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import GradientShaping functions
from gs_functions import *

# Output files
PER_FOLD_CSV = RESULTS_DIR / 'per_fold_results.csv'
SUMMARY_CSV = RESULTS_DIR / 'summary_results.csv'

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
SEED = 1337

def seed_everything(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# Training hyperparameters
BATCH_SIZE = 128
NUM_EPOCHS = 50
PATIENCE = 7
MAX_TRAIN_SLICES = 12000  # Training cap only (does not affect evaluation)

# Experimental grid
ARCHITECTURES = ['resnet18', 'densenet121']
CONDITIONS = ['raw', 'gs0', 'gs10', 'gs20', 'gs30', 'gs40', 'gs50']

# GS parameters
GS_BATCH_SIZE = 100
GS_ITERATIONS = 50

print("="*70)
print("OASIS CLINICAL UTILITY - CONFIGURATION")
print("="*70)
print(f"Device: {DEVICE}")
print(f"Seed: {SEED}")
print(f"Data dir: {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Patience: {PATIENCE}")
print(f"Max train slices: {MAX_TRAIN_SLICES}")
print(f"Architectures: {ARCHITECTURES}")
print(f"Conditions: {CONDITIONS}")
print("="*70)

OASIS CLINICAL UTILITY - CONFIGURATION
Device: cuda
Seed: 1337
Data dir: data/oasis/processed/utility
Results dir: results/oasis/utility
Batch size: 128
Max epochs: 50
Patience: 7
Max train slices: 12000
Architectures: ['resnet18', 'densenet121']
Conditions: ['raw', 'gs0', 'gs10', 'gs20', 'gs30', 'gs40', 'gs50']


In [2]:
# =============================================================================
# LOAD PREPROCESSED DATA & FOLDS
# =============================================================================

def load_utility_data(data_dir: Path) -> Tuple[np.ndarray, np.ndarray, List[str], List[int]]:
    """
    Load preprocessed utility data from band_data.pkl.
    
    Returns:
        images: (N, 224, 224) float32, normalized [0, 1]
        labels: (N,) int64, binary (0=healthy, 1=demented)
        subjects: list of subject IDs
        slice_nums: list of slice numbers
    """
    data_path = data_dir / 'band_data.pkl'
    
    with open(data_path, 'rb') as f:
        data = pickle.load(f)
    
    images = data['images']
    labels = data['labels']
    subjects = data['subjects']
    slice_nums = data['slice_nums']
    
    print(f"✓ Loaded data: {images.shape}")
    print(f"  Labels: {dict(zip(*np.unique(labels, return_counts=True)))}")
    print(f"  Subjects: {len(set(subjects))}")
    
    return images, labels, subjects, slice_nums


def load_folds(data_dir: Path) -> Dict:
    """Load pre-generated fold assignments."""
    folds_path = data_dir / 'folds.pkl'
    
    with open(folds_path, 'rb') as f:
        folds = pickle.load(f)
    
    print(f"✓ Loaded {len(folds)} folds")
    for k in range(len(folds)):
        fold = folds[f'fold_{k}']
        print(f"  Fold {k}: train={len(fold['train'])}, val={len(fold['val'])}, test={len(fold['test'])}")
    
    return folds


# Load data
print("\n[LOAD] Loading preprocessed data...")
IMAGES, LABELS, SUBJECTS, SLICE_NUMS = load_utility_data(DATA_DIR)
FOLDS = load_folds(DATA_DIR)

# Build subject -> indices mapping for fast lookup
SUBJECT_TO_INDICES = {}
for i, sid in enumerate(SUBJECTS):
    SUBJECT_TO_INDICES.setdefault(sid, []).append(i)

print(f"\n✓ Data ready: {len(IMAGES)} slices, {len(SUBJECT_TO_INDICES)} subjects")


[LOAD] Loading preprocessed data...
✓ Loaded data: (43927, 224, 224)
  Labels: {np.int64(0): np.int64(34162), np.int64(1): np.int64(9765)}
  Subjects: 347
✓ Loaded 5 folds
  Fold 0: train=221, val=56, test=70
  Fold 1: train=221, val=56, test=70
  Fold 2: train=222, val=56, test=69
  Fold 3: train=222, val=56, test=69
  Fold 4: train=222, val=56, test=69

✓ Data ready: 43927 slices, 347 subjects


In [3]:
# =============================================================================
# MODEL DEFINITION
# =============================================================================

def get_weights_enum(arch: str):
    """Get pretrained weights enum for architecture."""
    if arch == 'resnet18':
        return models.ResNet18_Weights.DEFAULT
    if arch == 'densenet121':
        return models.DenseNet121_Weights.DEFAULT
    raise ValueError(f"Unknown architecture: {arch}")


def get_model(arch: str = 'resnet18', weights=None) -> nn.Module:
    """
    Create model with 1-channel input and 2-class output.
    
    Args:
        arch: 'resnet18' or 'densenet121'
        weights: pretrained weights or None for scratch
    
    Returns:
        Model moved to DEVICE
    """
    is_pretrained = weights is not None
    
    if arch == 'resnet18':
        model = models.resnet18(weights=weights)
        
        # Adapt input conv to 1-channel
        old_conv = model.conv1
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        if is_pretrained:
            with torch.no_grad():
                model.conv1.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        else:
            nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')
        
        # Partial freeze for pretrained
        if is_pretrained:
            for name, param in model.named_parameters():
                if not name.startswith('layer3') and not name.startswith('layer4') and not name.startswith('fc'):
                    param.requires_grad = False
        
        # Stronger head regularization
        num_ftrs = model.fc.in_features
        head_linear = nn.Linear(num_ftrs, 2)
        nn.init.xavier_uniform_(head_linear.weight)
        nn.init.zeros_(head_linear.bias)
        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            head_linear
        )
    
    elif arch == 'densenet121':
        model = models.densenet121(weights=weights, memory_efficient=True)
        
        # Adapt input conv to 1-channel
        old_conv = model.features.conv0
        model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        if is_pretrained:
            with torch.no_grad():
                model.features.conv0.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        else:
            nn.init.kaiming_normal_(model.features.conv0.weight, mode='fan_out', nonlinearity='relu')
        
        # Partial freeze for pretrained
        if is_pretrained:
            for name, param in model.named_parameters():
                if not name.startswith('features.denseblock3') \
                   and not name.startswith('features.denseblock4') \
                   and not name.startswith('classifier'):
                    param.requires_grad = False
        
        # Stronger head regularization
        num_ftrs = model.classifier.in_features
        head_linear = nn.Linear(num_ftrs, 2)
        nn.init.xavier_uniform_(head_linear.weight)
        nn.init.zeros_(head_linear.bias)
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            head_linear
        )
    
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    
    return model.to(DEVICE)


print("✓ Model factory ready")

✓ Model factory ready


In [4]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def compute_class_weights(labels: torch.Tensor) -> torch.Tensor:
    """Compute balanced class weights from training labels."""
    y = labels.detach().cpu().numpy().astype(int)
    n0 = (y == 0).sum()
    n1 = (y == 1).sum()
    
    if n0 == 0 or n1 == 0:
        return torch.ones(2, dtype=torch.float32)
    
    w0 = (n0 + n1) / (2.0 * n0)
    w1 = (n0 + n1) / (2.0 * n1)
    
    return torch.tensor([w0, w1], dtype=torch.float32)


def aggregate_subject_probs(
    subject_ids: List[str],
    probs_pos: np.ndarray,
    labels: np.ndarray
) -> Tuple[List[str], np.ndarray, np.ndarray]:
    """
    Aggregate slice-level predictions to subject-level.
    
    Returns:
        subj_ids: list of unique subject IDs
        subj_probs: mean P(y=1) per subject
        subj_labels: ground truth label per subject
    """
    subj_to_probs = {}
    subj_to_label = {}
    
    for sid, p, y in zip(subject_ids, probs_pos, labels):
        sid = str(sid)
        subj_to_probs.setdefault(sid, []).append(float(p))
        
        if sid in subj_to_label and subj_to_label[sid] != int(y):
            raise ValueError(f"Label inconsistency for subject {sid}")
        subj_to_label[sid] = int(y)
    
    subj_ids = list(subj_to_probs.keys())
    subj_probs = np.array([np.mean(subj_to_probs[sid]) for sid in subj_ids], dtype=float)
    subj_labels = np.array([subj_to_label[sid] for sid in subj_ids], dtype=int)
    
    return subj_ids, subj_probs, subj_labels


def compute_confidence_metrics(subj_probs: np.ndarray, subj_labels: np.ndarray) -> Dict:
    """Compute confidence metrics by prediction correctness."""
    p = np.clip(subj_probs, 1e-12, 1.0 - 1e-12)
    y = subj_labels.astype(int)
    
    pred = (p >= 0.5).astype(int)
    correct_mask = (pred == y)
    wrong_mask = ~correct_mask
    
    # P(true class)
    p_true = np.where(y == 1, p, 1.0 - p)
    nll = -np.log(np.clip(p_true, 1e-12, 1.0))
    nll_pct = np.exp(-nll) * 100.0
    
    # Max softmax probability
    msp = np.maximum(p, 1.0 - p)
    
    def safe_mean(arr, mask):
        return float(np.mean(arr[mask])) if mask.sum() > 0 else float('nan')
    
    return {
        'n_subj': int(len(y)),
        'n_correct': int(correct_mask.sum()),
        'n_wrong': int(wrong_mask.sum()),
        'msp_correct_mean': safe_mean(msp, correct_mask),
        'msp_wrong_mean': safe_mean(msp, wrong_mask),
        'nll_correct_mean': safe_mean(nll, correct_mask),
        'nll_wrong_mean': safe_mean(nll, wrong_mask),
        'nllpct_correct_mean': safe_mean(nll_pct, correct_mask),
        'nllpct_wrong_mean': safe_mean(nll_pct, wrong_mask),
        'msp_all_mean': float(np.mean(msp)),
        'nll_all_mean': float(np.mean(nll)),
        'nllpct_all_mean': float(np.mean(nll_pct)),
    }


print("✓ Utility functions ready")

✓ Utility functions ready


In [5]:
# =============================================================================
# GS CACHE & TENSOR BUILDER
# =============================================================================

# Import GS functions (assumed to exist in gs_functions.py)
from gs_functions import GS_batch_image


def get_fold_indices(fold_idx: int, split: str, folds: Dict) -> np.ndarray:
    """
    Get slice indices for a fold split.
    
    Args:
        fold_idx: fold number (0-4)
        split: 'train', 'val', or 'test'
        folds: fold assignments dict
    
    Returns:
        Array of slice indices
    """
    fold = folds[f'fold_{fold_idx}']
    subject_ids = set(fold[split])
    
    indices = []
    for sid in subject_ids:
        if sid in SUBJECT_TO_INDICES:
            indices.extend(SUBJECT_TO_INDICES[sid])
    
    return np.array(sorted(indices), dtype=int)


def build_or_load_fold_tensors(
    fold_idx: int,
    split: str,
    condition: str,
    batch_size_gs: int = GS_BATCH_SIZE,
    ite: int = GS_ITERATIONS,
) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    """
    Build or load cached tensors for a fold/split/condition.
    
    Args:
        fold_idx: fold number (0-4)
        split: 'train', 'val', or 'test'
        condition: 'raw' or 'gsN' (e.g., 'gs10', 'gs20')
        batch_size_gs: batch size for GS processing
        ite: number of GS iterations
    
    Returns:
        X: (N, 1, 224, 224) float32 tensor
        y: (N,) long tensor
        subject_ids: list of subject IDs
    """
    # Cache path
    cache_path = CACHE_DIR / f'fold{fold_idx}_{split}_{condition}.pt'

    # Load from cache only if caching is enabled
    if CACHE_ENABLED and cache_path.exists():
        print(f"  📦 Loading cache: {cache_path.name}")
        obj = torch.load(cache_path, map_location='cpu', weights_only=False)
        return obj['x'], obj['y'], obj['sid']
    
    # Get indices for this fold/split
    indices = get_fold_indices(fold_idx, split, FOLDS)
    
    # Extract data
    X = IMAGES[indices].copy()  # (N, 224, 224)
    y = LABELS[indices].copy()
    sids = [SUBJECTS[i] for i in indices]
    
    print(f"  Building {condition} tensors: {X.shape}")
    
    # Apply GS if requested
    if condition.startswith('gs'):
        mask_pct = int(condition.replace('gs', '')) / 100.0
        print(f"  Applying GS (maskP={mask_pct}, ite={ite})...")
        X = GS_batch_image(X, batch_size=batch_size_gs, ite=ite, maskP=mask_pct).astype(np.float32)
    
    # Convert to tensors
    X = torch.from_numpy(X).unsqueeze(1).float()  # (N, 1, 224, 224)
    y = torch.from_numpy(y).long()
    
    # Save cache
    if CACHE_ENABLED:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({'x': X, 'y': y, 'sid': sids}, cache_path)
        print(f"  💾 Saved cache: {cache_path.name}")
    
    return X, y, sids


print("✓ GS cache & tensor builder ready")

✓ GS cache & tensor builder ready


In [6]:
# =============================================================================
# TRAINING & EVALUATION FUNCTION
# =============================================================================

def train_and_evaluate(
    arch: str,
    pretrained: bool,
    train_condition: str,
    fold_idx: int,
    train_tensors: torch.Tensor,
    train_labels: torch.Tensor,
    train_subject_ids: List[str],
    val_tensors: torch.Tensor,
    val_labels: torch.Tensor,
    val_subject_ids: List[str],
    attacker_type: str = 'adaptive',
    eval_condition: Optional[str] = None,
) -> None:
    """
    Train model and evaluate on validation set.
    
    Args:
        arch: model architecture
        pretrained: whether to use pretrained weights
        train_condition: condition used for training ('raw' or 'gsN')
        fold_idx: fold number
        train_tensors: training images (N, 1, 224, 224)
        train_labels: training labels (N,)
        train_subject_ids: list of subject IDs for training
        val_tensors: validation images
        val_labels: validation labels
        val_subject_ids: list of subject IDs for validation
        attacker_type: 'adaptive' or 'blind'
        eval_condition: condition used for evaluation (defaults to train_condition)
    """
    if eval_condition is None:
        eval_condition = train_condition
    
    init_str = 'pretrained' if pretrained else 'scratch'
    setting_name = f"{arch}_{init_str}_fold{fold_idx}_train-{train_condition}_eval-{eval_condition}_{attacker_type}"
    
    # Validate inputs
    assert train_tensors.ndim == 4 and train_tensors.shape[1:] == (1, 224, 224), \
        f"Invalid train shape: {train_tensors.shape}"
    assert train_condition == 'raw' or train_condition.startswith('gs'), \
        f"Invalid train_condition: {train_condition}"
    assert attacker_type in {'adaptive', 'blind'}, \
        f"Invalid attacker_type: {attacker_type}"
    
    # Checkpoint path (depends only on training condition)
    ckpt_dir = MODELS_DIR / arch / init_str / train_condition
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f'fold_{fold_idx}.pt'
    
    # Hyperparameters
    lr = 5e-5 if pretrained else 8e-5
    wd = 1e-4 if pretrained else 5e-4
    
    # Stats
    n_train_slices = int(train_labels.shape[0])
    n_val_slices = int(val_labels.shape[0])
    n_train_subjects = len(set(train_subject_ids))
    n_val_subjects = len(set(val_subject_ids))
    
    # Model
    weights = get_weights_enum(arch) if pretrained else None
    ckpt_exists = ckpt_path.exists()
    
    print("\n" + "="*80)
    print(f"🚀 {setting_name}")
    print(f"   LR={lr:.1e} | WD={wd:.1e} | Seed={SEED}")
    if ckpt_exists:
        print(f"   ✓ Checkpoint exists → SKIP TRAIN, EVAL ONLY")
    print("="*80)
    
    model = get_model(arch=arch, weights=weights)
    
    # Class weights
    class_weights = compute_class_weights(train_labels).to(DEVICE)
    
    # Loss functions
    criterion_train = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    criterion_val = nn.CrossEntropyLoss(weight=None, label_smoothing=0.0)
    
    # Optimizer & scheduler
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
    
    # Data loaders
    train_loader = DataLoader(
        TensorDataset(train_tensors, train_labels),
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(val_tensors, val_labels),
        batch_size=BATCH_SIZE, shuffle=False
    )
    
    best_val_auc = float('nan')
    best_epoch = float('nan')
    best_val_loss = float('nan')
    
    # Training loop
    if not ckpt_exists:
        best_val_auc = -1.0
        best_epoch = -1
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        use_amp = (DEVICE.type == 'cuda')
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        max_grad_norm = 1.0
        
        for epoch in range(NUM_EPOCHS):
            # Train
            model.train()
            running_loss = 0.0
            
            for imgs, y in train_loader:
                imgs, y = imgs.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                
                with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                    logits = model(imgs)
                    loss = criterion_train(logits, y)
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                
                running_loss += float(loss.item())
            
            # Validate (subject-level)
            model.eval()
            val_loss_sum = 0.0
            val_batches = 0
            probs_pos = []
            ys = []
            
            with torch.no_grad():
                for imgs, y in val_loader:
                    imgs, y = imgs.to(DEVICE), y.to(DEVICE)
                    logits = model(imgs)
                    
                    vloss = criterion_val(logits, y)
                    val_loss_sum += float(vloss.item())
                    val_batches += 1
                    
                    p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                    probs_pos.append(p)
                    ys.append(y.cpu().numpy())
            
            val_loss = val_loss_sum / max(val_batches, 1)
            probs_pos = np.concatenate(probs_pos)
            ys = np.concatenate(ys)
            
            _, subj_probs, subj_labels = aggregate_subject_probs(val_subject_ids, probs_pos, ys)
            subj_pred = (subj_probs >= 0.5).astype(int)
            subj_bacc = balanced_accuracy_score(subj_labels, subj_pred) if len(np.unique(subj_labels)) == 2 else float('nan')
            subj_auc = roc_auc_score(subj_labels, subj_probs) if len(np.unique(subj_labels)) == 2 else float('nan')
            
            scheduler.step(subj_auc if not np.isnan(subj_auc) else 0.0)
            cur_lr = optimizer.param_groups[0]['lr']
            
            print(f"  Epoch {epoch+1:02d} | trainLoss={running_loss/max(len(train_loader),1):.4f} "
                  f"| valLoss={val_loss:.4f} | valSubjAUC={subj_auc:.4f} | valSubjBAcc={subj_bacc:.4f} | lr={cur_lr:.2e}")
            
            # Early stopping
            if not np.isnan(subj_auc) and subj_auc > best_val_auc:
                best_val_auc = float(subj_auc)
                best_epoch = int(epoch + 1)
                best_val_loss = float(val_loss)
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= PATIENCE:
                print(f"  🛑 Early stopping at epoch {epoch+1} (best: epoch {best_epoch}, AUC={best_val_auc:.4f})")
                break
    
    # Final evaluation on checkpoint
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    
    probs_pos = []
    ys = []
    with torch.no_grad():
        for imgs, y in val_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            probs_pos.append(p)
            ys.append(y.numpy())
    
    probs_pos = np.concatenate(probs_pos)
    ys = np.concatenate(ys)
    
    _, subj_probs, subj_labels = aggregate_subject_probs(val_subject_ids, probs_pos, ys)
    
    subj_pred = (subj_probs >= 0.5).astype(int)
    cm = confusion_matrix(subj_labels, subj_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    subj_acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    subj_bacc = balanced_accuracy_score(subj_labels, subj_pred) if len(np.unique(subj_labels)) == 2 else float('nan')
    subj_auc = roc_auc_score(subj_labels, subj_probs) if len(np.unique(subj_labels)) == 2 else float('nan')
    
    conf = compute_confidence_metrics(subj_probs, subj_labels)
    
    print("\n" + "-"*80)
    print(f"📋 FINAL | {setting_name}")
    print(f"  Subjects: {conf['n_subj']} | Correct: {conf['n_correct']} | Wrong: {conf['n_wrong']}")
    print(f"  AUC={subj_auc:.4f} | Acc={subj_acc:.4f} | BAcc={subj_bacc:.4f} | Sens={sens:.4f} | Spec={spec:.4f}")
    print(f"  MSP: correct={conf['msp_correct_mean']:.4f} | wrong={conf['msp_wrong_mean']:.4f}")
    print(f"  NLL: correct={conf['nll_correct_mean']:.4f} | wrong={conf['nll_wrong_mean']:.4f}")
    print("-"*80)
    
    # Log results
    row = {
        'model': arch,
        'init': init_str,
        'train_condition': train_condition,
        'eval_condition': eval_condition,
        'attacker_type': attacker_type,
        'fold_id': fold_idx,
        'subject_auc': subj_auc,
        'subject_acc': subj_acc,
        'subject_bacc': subj_bacc,
        'sensitivity': sens,
        'specificity': spec,
        'msp_correct_mean': conf['msp_correct_mean'],
        'msp_wrong_mean': conf['msp_wrong_mean'],
        'nll_correct_mean': conf['nll_correct_mean'],
        'nll_wrong_mean': conf['nll_wrong_mean'],
        'nllpct_correct_mean': conf['nllpct_correct_mean'],
        'nllpct_wrong_mean': conf['nllpct_wrong_mean'],
        'n_train_subjects': n_train_subjects,
        'n_val_subjects': n_val_subjects,
        'n_train_slices': n_train_slices,
        'n_val_slices': n_val_slices,
        'lr': lr,
        'weight_decay': wd,
        'patience': PATIENCE,
        'num_epochs_max': NUM_EPOCHS,
        'epochs_trained': best_epoch,
        'best_val_auc': best_val_auc,
        'best_val_loss': best_val_loss,
        'seed': SEED,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'ckpt_path': str(ckpt_path),
    }
    
    pd.DataFrame([row]).to_csv(
        PER_FOLD_CSV,
        mode='a',
        header=not PER_FOLD_CSV.exists(),
        index=False,
    )
    print(f"✓ Logged → {PER_FOLD_CSV}")
    
    # Cleanup
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()


print("✓ Training & evaluation function ready")

✓ Training & evaluation function ready


In [7]:
# =============================================================================
# COMPLETION CHECK & RESULTS SUMMARY
# =============================================================================

def check_completion(arch: str, pretrained: bool, train_condition: str, fold_idx: int) -> bool:
    """Check if a specific experiment has already been completed."""
    init_str = 'pretrained' if pretrained else 'scratch'
    ckpt_path = MODELS_DIR / arch / init_str / train_condition / f'fold_{fold_idx}.pt'
    return ckpt_path.exists()


def get_results_summary() -> pd.DataFrame:
    """Get summary of all completed experiments."""
    if not PER_FOLD_CSV.exists():
        print("No results logged yet.")
        return None
    
    df = pd.read_csv(PER_FOLD_CSV)
    
    summary = (
        df.groupby(['model', 'init', 'train_condition', 'eval_condition', 'attacker_type'])
        [['subject_auc', 'subject_acc', 'subject_bacc']]
        .agg(['mean', 'std'])
        .round(4)
    )
    
    return summary


print("✓ Completion check & results summary ready")

✓ Completion check & results summary ready


## Main Execution Loop

In [8]:
# =============================================================================
# MAIN EXPERIMENT LOOP
# =============================================================================

print("\n" + "="*80)
print("🚀 STARTING OASIS UTILITY EXPERIMENTS")
print("="*80)

# Loop: pretrained first, then scratch
for pretrained in [True, False]:
    phase = "PRETRAINED" if pretrained else "SCRATCH"
    print(f"\n{'='*80}")
    print(f"📌 PHASE: {phase}")
    print(f"{'='*80}")
    
    for fold_idx in range(5):
        fold_name = f'fold_{fold_idx}'
        fold = FOLDS[fold_name]
        
        # Get indices
        train_idx = get_fold_indices(fold_idx, 'train', FOLDS)
        val_idx = get_fold_indices(fold_idx, 'val', FOLDS)
        
        # Apply training cap
        if len(train_idx) > MAX_TRAIN_SLICES:
            rng = np.random.default_rng(SEED + fold_idx)
            train_idx = rng.choice(train_idx, size=MAX_TRAIN_SLICES, replace=False)
            print(f"  ⚠ Capped training to {MAX_TRAIN_SLICES} slices")
        
        for condition in CONDITIONS:
            for arch in ARCHITECTURES:
                
                # Skip if already done
                if check_completion(arch, pretrained, condition, fold_idx):
                    print(f"⏩ Skip: fold={fold_idx} {arch} {condition} {'pretrained' if pretrained else 'scratch'}")
                    continue
                
                print(f"\n▶ RUNNING: fold={fold_idx} | {arch} | {'pretrained' if pretrained else 'scratch'} | {condition}")
                
                # Build/load tensors
                train_x, train_y, train_sid = build_or_load_fold_tensors(
                    fold_idx=fold_idx,
                    split='train',
                    condition=condition,
                )
                
                # Apply cap to loaded tensors if needed
                if len(train_x) > MAX_TRAIN_SLICES:
                    rng = np.random.default_rng(SEED + fold_idx)
                    cap_idx = rng.choice(len(train_x), size=MAX_TRAIN_SLICES, replace=False)
                    train_x = train_x[cap_idx]
                    train_y = train_y[cap_idx]
                    train_sid = [train_sid[i] for i in cap_idx]
                
                val_x, val_y, val_sid = build_or_load_fold_tensors(
                    fold_idx=fold_idx,
                    split='val',
                    condition=condition,
                )
                
                # Adaptive evaluation (train & eval on same condition)
                train_and_evaluate(
                    arch=arch,
                    pretrained=pretrained,
                    train_condition=condition,
                    fold_idx=fold_idx,
                    train_tensors=train_x,
                    train_labels=train_y,
                    train_subject_ids=train_sid,
                    val_tensors=val_x,
                    val_labels=val_y,
                    val_subject_ids=val_sid,
                    attacker_type='adaptive',
                    eval_condition=condition,
                )
                
                # Blind evaluation (train on GS, eval on raw) - only for GS conditions
                if condition != 'raw':
                    val_x_raw, val_y_raw, val_sid_raw = build_or_load_fold_tensors(
                        fold_idx=fold_idx,
                        split='val',
                        condition='raw',
                    )
                    
                    train_and_evaluate(
                        arch=arch,
                        pretrained=pretrained,
                        train_condition=condition,
                        fold_idx=fold_idx,
                        train_tensors=train_x,
                        train_labels=train_y,
                        train_subject_ids=train_sid,
                        val_tensors=val_x_raw,
                        val_labels=val_y_raw,
                        val_subject_ids=val_sid_raw,
                        attacker_type='blind',
                        eval_condition='raw',
                    )
                    
                    del val_x_raw, val_y_raw
                
                # Cleanup
                del train_x, train_y, val_x, val_y
                gc.collect()
                torch.cuda.empty_cache()

print("\n" + "="*80)
print("✓ ALL EXPERIMENTS COMPLETE")
print("="*80)

# Show summary
summary = get_results_summary()
if summary is not None:
    print("\n📊 RESULTS SUMMARY:")
    print(summary)


🚀 STARTING OASIS UTILITY EXPERIMENTS

📌 PHASE: PRETRAINED
  ⚠ Capped training to 12000 slices
⏩ Skip: fold=0 resnet18 raw pretrained
⏩ Skip: fold=0 densenet121 raw pretrained
⏩ Skip: fold=0 resnet18 gs0 pretrained
⏩ Skip: fold=0 densenet121 gs0 pretrained
⏩ Skip: fold=0 resnet18 gs10 pretrained
⏩ Skip: fold=0 densenet121 gs10 pretrained
⏩ Skip: fold=0 resnet18 gs20 pretrained
⏩ Skip: fold=0 densenet121 gs20 pretrained
⏩ Skip: fold=0 resnet18 gs30 pretrained
⏩ Skip: fold=0 densenet121 gs30 pretrained
⏩ Skip: fold=0 resnet18 gs40 pretrained
⏩ Skip: fold=0 densenet121 gs40 pretrained
⏩ Skip: fold=0 resnet18 gs50 pretrained
⏩ Skip: fold=0 densenet121 gs50 pretrained
  ⚠ Capped training to 12000 slices
⏩ Skip: fold=1 resnet18 raw pretrained
⏩ Skip: fold=1 densenet121 raw pretrained
⏩ Skip: fold=1 resnet18 gs0 pretrained
⏩ Skip: fold=1 densenet121 gs0 pretrained
⏩ Skip: fold=1 resnet18 gs10 pretrained
⏩ Skip: fold=1 densenet121 gs10 pretrained
⏩ Skip: fold=1 resnet18 gs20 pretrained
⏩ Skip: